# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Lecture 5: Limits and Risk

This notebook accompanies Chapter 3, Sections 3.2--3.4. We compare the
three required modes of convergence, illustrate the weak and strong laws of
large numbers, and simulate the correctly standardized central limit theorem.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

rng = np.random.default_rng(505)


## Three ways random variables can converge

Let $X_1,X_2,\ldots$ and $X$ be real-valued random variables.

- **In distribution:** $F_n(t)\to F(t)$ at every continuity point of
  the CDF $F$ of $X$. The variables may live on different probability
  spaces.
- **In probability:** on one common probability space,
  $\mathbb P(|X_n-X|>\varepsilon)\to0$ for every $\varepsilon>0$.
- **Almost surely:** on one common probability space,
  $\mathbb P(\{\omega:X_n(\omega)\to X(\omega)\})=1$.

Almost-sure convergence implies convergence in probability, which implies
convergence in distribution. The converses do not hold in general.


## A normal example on one probability space

Let $Z\sim\mathcal N(0,1)$ and define $X_n=Z/\sqrt n$ on the same
probability space. For every real value of $Z$, $X_n\to0$, so the
convergence is almost sure (and hence also in probability and distribution).
The distribution of $X_n$ is $\mathcal N(0,1/n)$: the second parameter
is the variance.

The limit is a point mass at zero, whose CDF jumps at zero. Convergence in
distribution is required only at continuity points; indeed
$F_n(0)=1/2$ for every $n$, while the limiting CDF equals 1 at zero.


In [ ]:
x_grid = np.linspace(-2, 2, 1000)
fig, ax = plt.subplots(figsize=(7, 3.8))
for n in (1, 5, 25, 100):
    ax.plot(x_grid, norm.cdf(x_grid, loc=0, scale=1 / np.sqrt(n)), label=f"n={n}")
limit_cdf = (x_grid >= 0).astype(float)
ax.plot(x_grid, limit_cdf, color="black", linestyle="--", label="point mass at 0")
ax.set(xlabel="t", ylabel="CDF")
ax.legend()
plt.show()


In [ ]:
z = rng.normal()
sample_sizes = np.arange(1, 501)
coupled_path = z / np.sqrt(sample_sizes)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(sample_sizes, coupled_path)
ax.axhline(0, color="black", linestyle="--")
ax.set(xlabel="n", ylabel="$X_n$", title=f"One coupled path, Z={z:.3f}")
plt.show()


## What the laws of large numbers say

One required weak-law version assumes pairwise independent random variables,
a common finite mean $\mu$, finite second moments, and a common finite
upper bound on their variances. It concludes
$\overline X_n\to\mu$ in probability.

For the strong law used in the notes, if $X_1,X_2,\ldots$ are i.i.d. and
$\mathbb E|X_1|<\infty$, then
$\overline X_n\to\mathbb E[X_1]$ almost surely. Independence and the
moment assumptions are hypotheses, not consequences of collecting repeated
observations.


In [ ]:
p = 0.30
bernoulli_path = rng.binomial(1, p, size=5000)
running_mean = np.cumsum(bernoulli_path) / np.arange(1, bernoulli_path.size + 1)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(np.arange(1, bernoulli_path.size + 1), running_mean, label="running sample mean")
ax.axhline(p, color="black", linestyle="--", label="$E[X_1]=0.3$")
ax.set(xlabel="n", ylabel=r"$\overline{X}_n$")
ax.legend()
plt.show()


A Cauchy random variable has no finite mean, so neither law above can be
invoked for i.i.d. Cauchy observations. A simulated path may be informative,
but it cannot repair the missing hypothesis or prove convergence or failure
of convergence.


In [ ]:
cauchy_path = rng.standard_cauchy(size=5000)
cauchy_running_mean = np.cumsum(cauchy_path) / np.arange(1, cauchy_path.size + 1)

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(np.arange(1, cauchy_path.size + 1), cauchy_running_mean)
ax.axhline(0, color="black", linestyle="--")
ax.set(xlabel="n", ylabel="running Cauchy mean", ylim=(-10, 10))
plt.show()


## Seeing the central limit theorem in a simulation

If $X_1,X_2,\ldots$ are i.i.d.,
$\mu=\mathbb E[X_1]$, and
$0<\sigma^2=\operatorname{Var}(X_1)<\infty$, then

$$
Z_n=\frac{\sqrt n(\overline X_n-\mu)}{\sigma}
\;\xrightarrow{d}\;\mathcal N(0,1).
$$

The normalization matters: $\operatorname{Var}(Z_n)=1$, whereas
$\operatorname{Var}(\overline X_n)=\sigma^2/n\to0$. The theorem does not
say that $\overline X_n$ converges to an $n$-dependent normal “limit.”


In [ ]:
repetitions = 10_000
sample_sizes = (2, 10, 50)
z_by_n = {}
for n in sample_sizes:
    observations = rng.exponential(scale=1.0, size=(repetitions, n))
    # Exponential(rate=1) has mu=1 and sigma=1.
    z_by_n[n] = np.sqrt(n) * (observations.mean(axis=1) - 1.0)

normal_grid = np.linspace(-4, 4, 500)
normal_density = np.exp(-normal_grid**2 / 2) / np.sqrt(2 * np.pi)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharex=True, sharey=True)
for ax, n in zip(axes, sample_sizes):
    ax.hist(z_by_n[n], bins=60, range=(-4, 4), density=True, alpha=0.55)
    ax.plot(normal_grid, normal_density, color="black")
    ax.set_title(f"n={n}")
    ax.set_xlabel("standardized mean")
axes[0].set_ylabel("density")
plt.tight_layout()
plt.show()


## Checkpoint: limits

1. For $U\sim\mathrm{Uniform}([0,1])$, let
   $X_n=\mathbf 1_{\{U\leq1/n\}}$. Prove directly that $X_n\to0$
   almost surely and in probability.
2. State exactly which weak-law, strong-law, and CLT hypotheses are available
   for i.i.d. Bernoulli variables. Which conclusion does each theorem give?
3. Repeat the CLT experiment for i.i.d. $\mathrm{Uniform}([0,1])$
   observations. Derive $\mu$ and $\sigma$ before writing code.


## Risk

This notebook accompanies Chapter 4, Section 4.1. We specify supervised
learning models, distinguish population and empirical risk, study squared
loss in a finite regression model, and compute a Bayes classifier exactly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(506)


## Risk in the population and in the sample

Let $(X,Y)$ have a specified joint distribution, let $\mathcal M$ be a
class of prediction rules $g$, and let
$\ell(y,g(x))\geq0$ be a loss. The **population risk** of a fixed rule is

$$
R(g)=\mathbb E[\ell(Y,g(X))].
$$

For an i.i.d. training sample $(X_i,Y_i)_{i=1}^n$, its **empirical risk**
is

$$
\widehat R_n(g)=\frac1n\sum_{i=1}^n\ell(Y_i,g(X_i)).
$$

The population distribution and hence $R(g)$ are usually unknown. If an
algorithm chooses $\widehat g$ from the training sample, then
$\widehat g$ is random before the sample is observed. Its training risk
and its population risk are different objects.


## A regression model we can calculate by hand

Let $X$ be uniform on $\{-1,1\}$. Independently, let
$\varepsilon$ be uniform on $\{-1,1\}$, and set
$Y=2X+\varepsilon$. Under squared loss, consider the class
$\mathcal M=\{g_a:g_a(x)=ax,\ a\in\mathbb R\}$.

The regression function is $r(x)=\mathbb E[Y\mid X=x]=2x$. Expanding the
square and using independence and $\mathbb E[\varepsilon]=0$ gives

$$
R(g_a)=\mathbb E[(Y-aX)^2]=(2-a)^2+1.
$$

The irreducible error is 1, and the unique best slope is $a=2$.


In [ ]:
# Verify the population risk by enumerating all four equally likely outcomes.
x_support = np.array([-1.0, 1.0])
noise_support = np.array([-1.0, 1.0])
outcomes = np.array([(x, 2 * x + error) for x in x_support for error in noise_support])


def exact_squared_risk(slope):
    losses = (outcomes[:, 1] - slope * outcomes[:, 0]) ** 2
    return losses.mean()


for slope in (0, 1, 2, 3):
    print(slope, exact_squared_risk(slope), (2 - slope) ** 2 + 1)


In [ ]:
# One i.i.d. training sample: empirical risk fluctuates around population risk.
n = 40
x_train = rng.choice(x_support, size=n)
noise_train = rng.choice(noise_support, size=n)
y_train = 2 * x_train + noise_train

slopes = np.linspace(-0.5, 4.5, 301)
population_risk = (2 - slopes) ** 2 + 1
empirical_risk = np.array([np.mean((y_train - slope * x_train) ** 2) for slope in slopes])

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(slopes, population_risk, label="population risk")
ax.plot(slopes, empirical_risk, label="empirical risk from one sample")
ax.axvline(2, color="black", linestyle="--", label="population minimizer")
ax.set(xlabel="slope a", ylabel="squared risk")
ax.legend()
plt.show()


## Finding the best rule for binary classification

Suppose $X\in\{a,b,c\}$ has probabilities $(1/4,1/2,1/4)$, and

$$
\eta(x)=\mathbb P(Y=1\mid X=x)
\quad\text{equals}\quad (0.1,0.5,0.8).
$$

Under 0--1 loss, predicting 0 at $x$ has conditional risk $\eta(x)$,
while predicting 1 has conditional risk $1-\eta(x)$. The Bayes rule picks
the smaller one. We follow the notes' tie convention and predict 0 when
$\eta(x)=1/2$.


In [ ]:
x_names = np.array(["a", "b", "c"])
p_x = np.array([0.25, 0.50, 0.25])
eta = np.array([0.10, 0.50, 0.80])
bayes_prediction = (eta > 0.5).astype(int)


def classification_risk(prediction):
    conditional_error = np.where(prediction == 1, 1 - eta, eta)
    return np.sum(p_x * conditional_error)


always_one = np.ones(3, dtype=int)
bayes_risk = classification_risk(bayes_prediction)
always_one_risk = classification_risk(always_one)
excess_identity = np.sum(
    p_x * np.abs(2 * eta - 1) * (always_one != bayes_prediction)
)

print(dict(zip(x_names, bayes_prediction)))
print("Bayes risk =", bayes_risk)
print("always-one risk =", always_one_risk)
print("risk difference =", always_one_risk - bayes_risk)
print("excess-risk identity =", excess_identity)


At $x=b$, both predictions have conditional error $1/2$. Therefore the
Bayes rule with the stated tie convention is one minimizer, but changing its
prediction only at $b$ gives another minimizer. Existence and uniqueness of
a risk minimizer are not automatic in general prediction classes.


## Recap

1. In the regression model, enlarge the class to
   $g_{a,b}(x)=ax+b$. Compute the population risk and find every minimizer.
2. In the classification model, compute the risk of always predicting 0 and
   verify the Bayes excess-risk identity.
3. Explain why minimizing empirical risk on the training data does not make
   the resulting training risk equal to population risk.
